# Sentiment Analysis with SVM

## Initial Implementation: Linear SVM model

### Importing the data

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the TSV file using pandas
training_data = pd.read_csv('training_sets/sentiment-topic/sentiment-topic-train.tsv', sep='\t', header=0)  
test_data = pd.read_csv('test_sets/sentiment-topic-test.tsv', sep='\t')

training_data

,sentence,sentiment,topic
0,Aggie is angela lansbury who carries pocketboo...,neutral,book
1,While this light murder mystery is laced with ...,neutral,book
2,"The setting, views described in the story are ...",neutral,book
3,Very light reading.,neutral,book
4,I did not expect this type of book to be in li...,positive,book
...,...,...,...
35773,Wut,neutral,sports
35774,Does turin have mosh tuesdays,neutral,sports
35775,Realistically we could probably fetch 40 45m f...,neutral,sports
35776,Decent amount for barnes too wouldn t be surpr...,positive,sports


In [6]:
print(len(training_data))
print("train:", training_data[['sentiment']].value_counts(sort=False))
training_data.head(3)

35778
train: sentiment
negative      9566
neutral       9819
positive     16393
Name: count, dtype: int64


,sentence,sentiment,topic
0,Aggie is angela lansbury who carries pocketboo...,neutral,book
1,While this light murder mystery is laced with ...,neutral,book
2,"The setting, views described in the story are ...",neutral,book


In [7]:
# Split the training data into training and validation set
# Paper stating 0.1% as development set: https://ieeexplore-ieee-org.vu-nl.idm.oclc.org/document/9206796
train, dev = train_test_split(training_data, test_size=0.1, random_state=0, 
                               stratify=training_data[['sentiment']])

## Vectorizing with TD-IDF

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk

# Create feature vectors with TD-IDF
vectorizer = TfidfVectorizer(min_df = 2, # Which value? Paper ... (now taken from assignment)
                             tokenizer=nltk.word_tokenize # we use the nltk tokenizer
                            )
# Training 
train_vectors = vectorizer.fit_transform(train['sentence'])
# Validation
dev_vectors = vectorizer.transform(dev['sentence'])  
# Test
test_vectors = vectorizer.transform(test_data['sentence'])

c:\Users\Sandy\anaconda3\envs\textmining\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [41]:
from sklearn import svm
from sklearn.metrics import classification_report

# Perform classification with SVM, kernel=linear
svm_classifier = svm.SVC(kernel='linear', C=1, gamma=0.01) # Which kernel? Paper...
svm_classifier.fit(train_vectors, train['sentiment'])

# Predict the development set
dev_prediction = svm_classifier.predict(dev_vectors)
report_development = classification_report(dev['sentiment'], dev_prediction, target_names=training_data['sentiment'].unique())
print("Report training + development set \n")
print(report_development)

Report training + development set 

              precision    recall  f1-score   support

     neutral       0.64      0.60      0.62       957
    positive       0.60      0.50      0.55       982
    negative       0.71      0.80      0.75      1639

    accuracy                           0.67      3578
   macro avg       0.65      0.63      0.64      3578
weighted avg       0.66      0.67      0.66      3578



In [42]:
from scipy.sparse import vstack

# Combine the training and development set
complete_training_data = vstack([train_vectors,dev_vectors])
complete_training_labels = pd.concat([train['sentiment'],dev['sentiment']])

# Train model with complete training data
svm_classifier.fit(complete_training_data, complete_training_labels)

# Predict the test set
test_prediction = svm_classifier.predict(test_vectors)
report = classification_report(test_data['sentiment'], test_prediction, target_names=training_data['sentiment'].unique())
print("Report test set \n")
print(report)

Report test set 

              precision    recall  f1-score   support

     neutral       0.62      0.83      0.71         6
    positive       0.00      0.00      0.00         6
    negative       0.50      0.83      0.62         6

    accuracy                           0.56        18
   macro avg       0.38      0.56      0.45        18
weighted avg       0.38      0.56      0.45        18



c:\Users\Sandy\anaconda3\envs\textmining\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Sandy\anaconda3\envs\textmining\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Sandy\anaconda3\envs\textmining\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(r

#### Transition from Linear to RBF Kernel SVM

The initial linear SVM implementation showed poor performance on our sentiment analysis task, particularly failing to identify positive sentiment instances (0.00 precision and recall). This limitation stems from linear SVM's inability to capture complex, non-linear relationships between words and sentiment expressions.

Sentiment language often contains nuanced patterns that aren't linearly separable in the feature space. For example, negations, intensifiers, and context-dependent sentiment require more sophisticated decision boundaries. As noted in lecture 4, while SVM is effective for sentiment classification, choosing the right kernel is crucial.

We've therefore implemented an RBF (Radial Basis Function) kernel SVM, which can model non-linear relationships by implicitly mapping features to a higher-dimensional space. Additionally, we've added class weighting to address class imbalance and expanded our feature representation to include bigrams, capturing more context than individual words alone.

## Final Implementation: RBF Kernel SVM

1. Feature Extraction with TF-IDF Vectorization <br>
To transform our text data into numerical features suitable for machine learning, we implement TF-IDF (Term Frequency-Inverse Document Frequency) vectorization with enhanced features. 

Key features of our vectorization approach:

* TF-IDF weighting: Reduces impact of common words while emphasizing distinctive terms
* NLTK tokenization: Provides linguistic-aware word segmentation
* N-gram range (1,2): Captures both individual words and adjacent word pairs, helping identify sentiment in phrases like "not good" or "very happy"
* Minimum document frequency: Filters out extremely rare terms to reduce noise and overfitting

In [9]:
vectorizer = TfidfVectorizer(min_df = 2, # Which value? Paper ... (now taken from assignment)
                             tokenizer=nltk.word_tokenize, # we use the nltk tokenizer
                             ngram_range=(1,2) #Include unigrams and bigrams
                            )

# Training 
train_vectors = vectorizer.fit_transform(train['sentence'])
# Validation
dev_vectors = vectorizer.transform(dev['sentence'])  
# Test
test_vectors = vectorizer.transform(test_data['sentence'])


2. SVM training <br>
The RBF kernel allows the SVM to find non-linear decision boundaries in the feature space, which is critical for capturing complex sentiment patterns. The 'balanced' class weight option automatically adjusts weights inversely proportional to class frequencies, addressing our dataset's imbalance. We first validate our model on the development set to assess its performance before final testing.

In [10]:
from sklearn import svm
from sklearn.metrics import classification_report

# Perform classification with SVM, kernel=rbf
svm_classifier = svm.SVC(
    kernel='rbf', #Kernel SVM
    C=1,
    gamma='scale',
    class_weight='balanced' #Class weighting to handle imbalance
)
svm_classifier.fit(train_vectors, train['sentiment'])

# Predict the development set
dev_prediction = svm_classifier.predict(dev_vectors)
report_development = classification_report(dev['sentiment'], dev_prediction, target_names=training_data['sentiment'].unique(), zero_division=0)
print("Report training + development set \n")
print(report_development)

Report training + development set 

              precision    recall  f1-score   support

     neutral       0.62      0.61      0.62       957
    positive       0.56      0.56      0.56       982
    negative       0.73      0.75      0.74      1639

    accuracy                           0.66      3578
   macro avg       0.64      0.64      0.64      3578
weighted avg       0.66      0.66      0.66      3578



3. Evaluation on test set <br>

After training on the combined dataset, we evaluate on the held-out test set using the same metrics as the BERT model for direct comparison. The error rate calculation provides a straightforward measure of overall classification errors.

In [15]:
print(len(complete_training_labels))

35778


In [11]:
from scipy.sparse import vstack

# Combine the training and development set
complete_training_data = vstack([train_vectors,dev_vectors])
complete_training_labels = pd.concat([train['sentiment'],dev['sentiment']])

# Train model with complete training data
svm_classifier.fit(complete_training_data, complete_training_labels)

# Predict the test set
test_prediction = svm_classifier.predict(test_vectors)
report = classification_report(test_data['sentiment'], test_prediction, target_names=training_data['sentiment'].unique(), zero_division=0)

# Calculate error rate
error_rate = 1 - (test_prediction == test_data['sentiment'].values).mean()
print(f"Error Rate: {error_rate:.4f}")

print("Report test set \n")
print(report)

Error Rate: 0.2778
Report test set 

              precision    recall  f1-score   support

     neutral       0.55      1.00      0.71         6
    positive       1.00      0.50      0.67         6
    negative       1.00      0.67      0.80         6

    accuracy                           0.72        18
   macro avg       0.85      0.72      0.72        18
weighted avg       0.85      0.72      0.72        18



In [17]:
import pickle

# Save SVM model
with open('SVM_rbf.pkl','wb') as f:
    pickle.dump(svm_classifier,f)

# Save TD-IDF vectorizer
with open('vectorizer_rbf.pkl','wb') as f:
    pickle.dump(vectorizer,f)

## Quantitative Analysis of RBF Kernel SVM

Examining class-specific performance reveals interesting patterns. The model achieves perfect precision (1.00) for both positive and negative sentiment classes, indicating high confidence in these predictions. However, recall varies considerably across classes: 100% for neutral samples, but only 50% for positive samples and 67% for negative ones. This suggests the model is conservative in assigning positive and negative labels, only doing so when highly confident.

The neutral class shows a different pattern, with lower precision (0.55) but perfect recall (1.00), indicating the model tends to classify borderline cases as neutral. This creates a precision-recall tradeoff that's reflected in the F1-scores, which range from 0.67 for positive sentiment to 0.80 for negative sentiment.

The macro average precision (0.85) significantly exceeds macro average recall (0.72), confirming the model's tendency toward precision at the expense of recall for non-neutral classes. This performance profile suggests the RBF kernel SVM has effectively learned to identify clear sentiment signals but remains cautious with ambiguous expressions—a reasonable approach for applications where false positives are more problematic than false negatives.

Accuracy is not taken into account as there is substantial class imbalance in the data.